# Phase 2a — SFT with Self-Generated CoT (STaR Approach)
## NVIDIA Nemotron Reasoning Challenge

**Method:** STaR (Self-Taught Reasoner, Zelikman 2022)
- Phase 1 adapter acts as **teacher**: generates `<think>...</think>` reasoning chains
- Same Phase 1 adapter is **student warm-start**: training init (1 epoch faster convergence)
- **Quality filter**: only keep examples where the model's reasoning leads to the CORRECT answer
- Result: ~80-90% of generated chains are kept (verified correct reasoning)

**10-Hour GPU Budget (~9.3h actual):**
| Step | Time |
|---|---|
| Setup + model load | ~12 min |
| Batched CoT generation (all non-roman, batch=2) | ~7.0 h |
| Training (2 epochs, warm start) | ~1.5 h |
| Eval (5% split, competition metric) | ~0.8 h |
| Save + zip | ~5 min |

**Required Kaggle Dataset Input:**
Add `submission.zip` from Phase 1 as a Kaggle Dataset named **`nemotron-phase1-adapter`**.
- Path in notebook: `/kaggle/input/nemotron-phase1-adapter/submission.zip`
- Update `PHASE1_ADAPTER_DATASET` in Config cell if dataset name differs.


## Cell 1 — Environment Setup

> Run first. Fixes Triton read-only FS issue inherited from Phase 1.

In [8]:
import site, os, sys, shutil, subprocess, time, zipfile, json, re, math
from pathlib import Path

# ── Triton fix (identical to Phase 1) ─────────────────────────────────────────
CUTLASS_PATH = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(CUTLASS_PATH)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

read_only_triton = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton"
writable_dir     = "/kaggle/working/python_packages"
writable_triton  = os.path.join(writable_dir, "triton")

if not os.path.exists(writable_triton):
    os.makedirs(writable_dir, exist_ok=True)
    print("Copying triton to writable dir...")
    shutil.copytree(read_only_triton, writable_triton)

bin_dir = os.path.join(writable_triton, "backends", "nvidia", "bin")
if os.path.exists(bin_dir):
    for root, dirs, files in os.walk(bin_dir):
        for f in files:
            os.chmod(os.path.join(root, f), 0o755)
    print("Triton binaries: permissions granted.")

if writable_dir not in sys.path:
    sys.path.insert(0, writable_dir)

print("Environment setup complete.")


Copying triton to writable dir...
Triton binaries: permissions granted.
Environment setup complete.


## Cell 2 — Imports

In [9]:
import polars as pl
import pandas as pd
import numpy as np
import torch

import kagglehub
import mamba_ssm  # noqa — must precede transformers for Nemotron

from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from datasets import Dataset

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB VRAM")


/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


PyTorch : 2.12.0.dev20260324+cu128
CUDA    : True
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition  102.0 GB VRAM


## Cell 3 — Configuration

All knobs in one place.

In [13]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = "/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge"
TRAIN_PATH  = f"{DATA_DIR}/train.csv"
OUTPUT_DIR  = "/kaggle/working"
ADAPTER_DIR = f"{OUTPUT_DIR}/phase2a_adapter"

# ── Phase 1 adapter ────────────────────────────────────────────────────────────
# Kaggle auto-unpacks datasets — point directly at the dataset directory.
# Files (adapter_model.safetensors, adapter_config.json, tokenizer*, etc.)
# are already available at this path without any extraction step.
PHASE1_ADAPTER_DIR = "/kaggle/input/datasets/lokeshvns/nemotron-phase1-adapter"

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_HANDLE = "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"

# ── CoT Generation (STaR) ─────────────────────────────────────────────────────
# Categories to generate CoT for (hardest → easiest; roman_numerals skipped)
COT_CATEGORIES     = ["other", "binary", "integer_math", "word_sequence", "decimal_math"]
COT_MAX_NEW_TOKENS = 300       # thinking chain length; keep moderate for speed
COT_TEMPERATURE    = 0.7       # slightly creative — diverse reasoning
COT_TOP_P          = 0.9
COT_GEN_BATCH_SIZE = 2         # batched generation: ~2x speedup on Mamba SSM
CHECKPOINT_EVERY   = 200       # save partial CoT every N examples (crash recovery)
COT_CHECKPOINT     = f"{OUTPUT_DIR}/cot_checkpoint.json"

# ── Training ──────────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 1024          # CoT chains fit in 1024 (avg ~400 tokens)
BATCH_SIZE     = 4             # reduced from Phase 1's 8 (longer sequences)
GRAD_ACCUM     = 4             # effective batch = 16
LEARNING_RATE  = 2e-5          # lower — warm-starting from Phase 1 adapter
NUM_EPOCHS     = 2             # 2 epochs sufficient with warm start
LR_SCHEDULER   = "cosine"
WARMUP_RATIO   = 0.03          # shorter warmup — already warm
RANDOM_SEED    = 42

# ── Eval ──────────────────────────────────────────────────────────────────────
EVAL_SPLIT          = 0.05     # 5% ~ 475 examples (~50 min)
EVAL_MAX_NEW_TOKENS = 1024     # match competition (allow full reasoning)
EVAL_TEMPERATURE    = 1.0      # match competition
EVAL_TOP_P          = 1.0      # match competition

print("Configuration loaded.")
print(f"  CoT categories       : {COT_CATEGORIES}")
print(f"  CoT batch size       : {COT_GEN_BATCH_SIZE}")
print(f"  CoT max tokens       : {COT_MAX_NEW_TOKENS}")
print(f"  Checkpoint every     : {CHECKPOINT_EVERY}")
print(f"  Training seq length  : {MAX_SEQ_LENGTH}")
print(f"  Training LR          : {LEARNING_RATE}")
print(f"  Training epochs      : {NUM_EPOCHS}")


Configuration loaded.
  CoT categories       : ['other', 'binary', 'integer_math', 'word_sequence', 'decimal_math']
  CoT batch size       : 2
  CoT max tokens       : 300
  Checkpoint every     : 200
  Training seq length  : 1024
  Training LR          : 2e-05
  Training epochs      : 2


## Cell 4 — Extract Phase 1 Adapter

In [11]:
print(f"Verifying Phase 1 adapter at: {PHASE1_ADAPTER_DIR}")

# Kaggle auto-unpacks datasets — no zip extraction needed.
# The adapter files are directly available in the dataset input directory.
adapter_dir = Path(PHASE1_ADAPTER_DIR)
if not adapter_dir.exists():
    raise FileNotFoundError(
        f"Phase 1 adapter directory not found:\n  {PHASE1_ADAPTER_DIR}\n\n"
        "Steps to fix:\n"
        "  1. Go to Kaggle → Datasets → New Dataset\n"
        "  2. Upload your Phase 1 submission.zip — Kaggle will auto-unpack it\n"
        "  3. Title the dataset 'nemotron-phase1-adapter'\n"
        "  4. Add it as input to this notebook\n"
        "  5. Or update PHASE1_ADAPTER_DIR in the Config cell"
    )

# List all files in the adapter directory
files = sorted(adapter_dir.iterdir())
print(f"Found {len(files)} files in adapter directory:")
for fp in files:
    kb = fp.stat().st_size / 1024
    print(f"  {fp.name:<45} {kb:>10.1f} KB")

# Verify required files exist
required = ["adapter_config.json"]
missing  = [r for r in required if not (adapter_dir / r).exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

with open(adapter_dir / "adapter_config.json") as f:
    cfg = json.load(f)
print(f"\nPhase 1 LoRA: rank={cfg.get('r')}  alpha={cfg.get('lora_alpha')}")
print(f"  Modules : {cfg.get('target_modules')}")
print("\n✅ Phase 1 adapter ready (no extraction needed — Kaggle pre-unpacked).")


Verifying Phase 1 adapter at: /kaggle/input/datasets/lokeshvns/nemotron-phase1-adapter
Found 6 files in adapter directory:
  README.md                                            5.2 KB
  adapter_config.json                                  1.0 KB
  adapter_model.safetensors                      3439795.2 KB
  chat_template.jinja                                 10.3 KB
  tokenizer.json                                   16677.3 KB
  tokenizer_config.json                                0.4 KB

Phase 1 LoRA: rank=32  alpha=64
  Modules : .*\.(in_proj|out_proj|up_proj|down_proj)$

✅ Phase 1 adapter ready (no extraction needed — Kaggle pre-unpacked).


## Cell 5 — Load & Categorise Training Data

In [14]:
def classify_puzzle(prompt: str, answer: str) -> str:
    a = str(answer).strip()
    if re.fullmatch(r'[01]{4,}', a):                                            return "binary"
    if any(k in prompt.lower() for k in ["binary","bit","xor","shift","bitwise"]): return "binary"
    if re.fullmatch(r'[IVXLCDM]+', a):                                          return "roman_numerals"
    if re.fullmatch(r'-?\d+', a):                                               return "integer_math"
    if re.fullmatch(r'-?\d+\.\d+', a):                                       return "decimal_math"
    words = a.split()
    if len(words) >= 2 and all(re.fullmatch(r'[a-zA-Z]+', w) for w in words):  return "word_sequence"
    return "other"

# ── Competition metric (copied verbatim — no imports from local paths) ─────────
def extract_final_answer(text):
    if text is None: return 'NOT_FOUND'
    boxed_starts = list(re.finditer(r'\\boxed\{', text))
    matches = []
    for i, m in enumerate(boxed_starts):
        start = m.end()
        end   = boxed_starts[i+1].start() if i+1 < len(boxed_starts) else len(text)
        seg   = text[start:end]
        lb    = seg.rfind('}')
        matches.append(seg[:lb] if lb != -1 else seg)
    if matches:
        ne = [m.strip() for m in matches if m.strip()]
        return ne[-1] if ne else matches[-1].strip()
    for pat in [r'The final answer is:\s*([^\n]+)',
                r'Final answer is:\s*([^\n]+)',
                r'final answer\s*[:：]\s*([^\n]+)']:
        found = re.findall(pat, text, re.IGNORECASE)
        if found: return found[-1].strip()
    nums = re.findall(r'-?\d+(?:\.\d+)?', text)
    if nums: return nums[-1]
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else 'NOT_FOUND'

def verify(stored, predicted):
    stored, predicted = stored.strip(), predicted.strip()
    if re.fullmatch(r'[01]+', stored): return predicted.lower() == stored.lower()
    try:    return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except: return predicted.lower() == stored.lower()

# ── Load & split ───────────────────────────────────────────────────────────────
df = pl.read_csv(TRAIN_PATH).to_pandas()
df["category"] = [classify_puzzle(r["prompt"], r["answer"]) for _, r in df.iterrows()]

np.random.seed(RANDOM_SEED)
idx    = np.random.permutation(len(df))
n_eval = int(len(df) * EVAL_SPLIT)
eval_df  = df.iloc[idx[:n_eval]].reset_index(drop=True)
train_df = df.iloc[idx[n_eval:]].reset_index(drop=True)

print(f"Dataset: {len(df)} total  →  {len(train_df)} train  |  {len(eval_df)} eval")
print()
print(f"{'Category':<20} {'Train':>6} {'Eval':>6}  CoT?")
print("-" * 42)
for cat in sorted(df["category"].unique()):
    tr = (train_df["category"] == cat).sum()
    ev = (eval_df["category"]  == cat).sum()
    cot = "✓ generate" if cat in COT_CATEGORIES else "✗ skip (100% P1)"
    print(f"  {cat:<18} {tr:>6} {ev:>6}  {cot}")
print("-" * 42)
cot_eligible = (train_df["category"].isin(COT_CATEGORIES)).sum()
print(f"  CoT generation target: {cot_eligible} examples")

# Rough time estimate
secs_per_example = 8  # conservative estimate for Mamba SSM batch=2
est_hours = cot_eligible * secs_per_example / COT_GEN_BATCH_SIZE / 3600
print(f"  Estimated generation time: ~{est_hours:.1f}h (batch={COT_GEN_BATCH_SIZE}, ~{secs_per_example}s/example)")


Dataset: 9500 total  →  9025 train  |  475 eval

Category              Train   Eval  CoT?
------------------------------------------
  binary               1885    108  ✓ generate
  decimal_math         3022    169  ✓ generate
  integer_math          668     18  ✓ generate
  other                 823     45  ✓ generate
  roman_numerals       1496     80  ✗ skip (100% P1)
  word_sequence        1131     55  ✓ generate
------------------------------------------
  CoT generation target: 7529 examples
  Estimated generation time: ~8.4h (batch=2, ~8s/example)


## Cell 6 — Load Nemotron-30B + Phase 1 Adapter

> STaR approach: Phase 1 adapter is both the *teacher* (CoT generation) and the *student starting point* (training init).

In [15]:
print("Downloading Nemotron-30B (cached after first run)...")
MODEL_PATH = kagglehub.model_download(MODEL_HANDLE)
print(f"Model path: {MODEL_PATH}\n")

# ── Tokenizer (load from adapter dir — has chat_template.jinja) ────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(PHASE1_ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Left-padding for batched generation (causal LM inference)
tokenizer.padding_side = "left"
print(f"Vocab size     : {tokenizer.vocab_size}")
print(f"Pad token      : {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")
print(f"Padding side   : {tokenizer.padding_side} (correct for batched generation)")

# ── Base model ─────────────────────────────────────────────────────────────────
print("\nLoading base model (bf16, device_map=auto)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16,
    offload_folder="offload_dir",
)
model.config.use_cache = False

# ── Attach Phase 1 LoRA adapter ────────────────────────────────────────────────
print("\nAttaching Phase 1 adapter (teacher + student init)...")
model = PeftModel.from_pretrained(model, PHASE1_ADAPTER_DIR, is_trainable=True)
model.print_trainable_parameters()

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / 1e9
        rsrvd = torch.cuda.memory_reserved(i)  / 1e9
        print(f"  GPU {i}: {alloc:.2f} GB alloc / {rsrvd:.2f} GB reserved")

print("\n✅ Model + Phase 1 adapter loaded. Ready for CoT generation.")


Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1

Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Vocab size     : 131072
Pad token      : '<|im_end|>' (id=11)
Padding side   : left (correct for batched generation)

Loading base model (bf16, device_map=auto)...


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]


Attaching Phase 1 adapter (teacher + student init)...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116
  GPU 0: 66.69 GB alloc / 70.45 GB reserved

✅ Model + Phase 1 adapter loaded. Ready for CoT generation.


## Cell 7 — CoT Generation Helpers

In [16]:
# ── Chat template helper ──────────────────────────────────────────────────────
def apply_template(messages, add_generation_prompt=False):
    """Apply Nemotron chat template with enable_thinking. Falls back gracefully."""
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:    return tokenizer.apply_chat_template(messages, enable_thinking=True, **kw)
    except TypeError: return tokenizer.apply_chat_template(messages, **kw)


def build_cot_prompt(prompt: str, answer: str) -> str:
    """Backsolved prompt: give the correct answer, ask model to explain reasoning.
    
    This is the STaR backsolved-rationalization strategy.
    Giving the answer makes generation much more reliable than cold-solving.
    """
    user_content = (
        f"{prompt}\n\n"
        f"The correct answer is: {answer}\n\n"
        "Show your step-by-step reasoning that identifies the hidden rule from the "
        "examples above and applies it to reach this answer. Be specific about the "
        "pattern you found."
    )
    return apply_template([{"role": "user", "content": user_content}],
                          add_generation_prompt=True)


def extract_think_block(text: str) -> str:
    """Extract <think>...</think> content, or fall back to full output."""
    m = re.search(r'<think>\s*(.*?)\s*</think>', text, re.DOTALL)
    if m:
        return m.group(1).strip()
    # Fallback: everything before \boxed{ or last 2/3 of text
    bp = text.find('\\boxed{')
    if bp > 30:
        return text[:bp].strip()
    return text.strip()


def star_filter(cot_text: str, answer: str) -> bool:
    """STaR quality filter: keep CoT only if extracted answer matches ground truth.
    
    This prevents training on hallucinated or wrong reasoning chains.
    Expected ~80-90% pass rate for backsolved prompts.
    """
    extracted = extract_final_answer(cot_text)
    # Also check if the last boxed answer in the think block matches
    return verify(str(answer), extracted)


print("CoT generation helpers defined ✅")
print()
print("STaR pipeline:")
print("  1. Backsolved prompt: puzzle + known answer → model explains reasoning")
print("  2. Quality filter   : only keep CoT where reasoning → correct answer")
print("  3. Train on filtered CoT: model learns verified reasoning patterns")


CoT generation helpers defined ✅

STaR pipeline:
  1. Backsolved prompt: puzzle + known answer → model explains reasoning
  2. Quality filter   : only keep CoT where reasoning → correct answer
  3. Train on filtered CoT: model learns verified reasoning patterns


## Cell 8 — Batched CoT Generation (STaR)

**Batching strategy for Mamba SSM:**
- Mamba has O(1) state generation (no KV-cache quadratic growth)
- Batch=2 gives nearly 2× throughput vs single-sample
- Left-padding ensures all inputs in a batch start from the same position
- `pad_to_multiple_of=8` aligns to tensor cores

**Checkpoint saves** every 200 examples — safe to restart from partial progress.


In [ ]:
# ── Resume from checkpoint if exists ─────────────────────────────────────────
if Path(COT_CHECKPOINT).exists():
    with open(COT_CHECKPOINT) as f:
        cot_data = json.load(f)
    already_done = {d["id"] for d in cot_data}
    print(f"Resumed from checkpoint: {len(cot_data)} examples already generated.")
else:
    cot_data     = []
    already_done = set()
    print("No checkpoint found. Starting fresh.")

# ── Build generation queue ─────────────────────────────────────────────────────
gen_queue = train_df[train_df["category"].isin(COT_CATEGORIES)].reset_index(drop=True)
gen_queue = gen_queue[~gen_queue["id"].astype(str).isin(already_done)].reset_index(drop=True)

total_eligible = (train_df["category"].isin(COT_CATEGORIES)).sum()
print(f"\nGeneration queue: {len(gen_queue)} remaining  "
      f"(of {total_eligible} total eligible)")
print(f"Batch size: {COT_GEN_BATCH_SIZE}  |  Max tokens: {COT_MAX_NEW_TOKENS}")
print()

# ── Statistics trackers ────────────────────────────────────────────────────────
generated  = 0          # kept after STaR filter
rejected   = 0          # failed STaR quality filter
errors     = 0          # generation errors
cat_stats  = {}         # per-category counters
run_start  = time.time()

model.eval()

# ── Batch iteration ────────────────────────────────────────────────────────────
for batch_start in range(0, len(gen_queue), COT_GEN_BATCH_SIZE):
    batch = gen_queue.iloc[batch_start : batch_start + COT_GEN_BATCH_SIZE]

    # Build prompts for each example in the batch
    prompts = [build_cot_prompt(row["prompt"], str(row["answer"]))
               for _, row in batch.iterrows()]

    # Tokenise with left-padding (required for batched causal LM generation)
    try:
        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH - COT_MAX_NEW_TOKENS,  # leave room for output
        ).to(model.device)

        with torch.no_grad():
            out_ids = model.generate(
                **enc,
                max_new_tokens=COT_MAX_NEW_TOKENS,
                do_sample=True,
                temperature=COT_TEMPERATURE,
                top_p=COT_TOP_P,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        input_len = enc["input_ids"].shape[1]

        for i, (_, row) in enumerate(batch.iterrows()):
            new_tokens  = out_ids[i][input_len:]
            raw_output  = tokenizer.decode(new_tokens, skip_special_tokens=True)
            cot_text    = extract_think_block(raw_output)

            # ── STaR quality filter ─────────────────────────────────────────────
            cat = str(row["category"])
            if cat not in cat_stats:
                cat_stats[cat] = {"kept": 0, "rejected": 0}

            if len(cot_text) < 30 or not star_filter(raw_output + " " + cot_text, str(row["answer"])):
                rejected += 1
                cat_stats[cat]["rejected"] += 1
                continue

            cot_data.append({
                "id"      : str(row["id"]),
                "prompt"  : row["prompt"],
                "answer"  : str(row["answer"]),
                "category": cat,
                "cot"     : cot_text,
            })
            generated += 1
            cat_stats[cat]["kept"] += 1

    except Exception as e:
        errors += len(batch)
        print(f"  [err] batch at {batch_start}: {type(e).__name__}: {e}")
        continue

    # ── Checkpoint save ────────────────────────────────────────────────────────
    total_processed = generated + rejected + errors
    if total_processed % CHECKPOINT_EVERY < COT_GEN_BATCH_SIZE:
        with open(COT_CHECKPOINT, "w") as f:
            json.dump(cot_data, f)

    # ── Progress report every 100 kept examples ────────────────────────────────
    if generated % 100 == 0 and generated > 0:
        elapsed   = (time.time() - run_start) / 60
        rate_kept = generated / elapsed
        pct_done  = 100 * total_processed / len(gen_queue)
        keep_rate = 100 * generated / max(generated + rejected, 1)
        print(f"  [{generated:5d} kept | {rejected:4d} rejected | {errors:3d} err]  "
              f"{pct_done:.1f}% done  {elapsed:.1f}m  rate={rate_kept:.1f}/min  "
              f"keep={keep_rate:.0f}%")

# ── Final checkpoint ───────────────────────────────────────────────────────────
with open(COT_CHECKPOINT, "w") as f:
    json.dump(cot_data, f)

elapsed_total = (time.time() - run_start) / 60
keep_rate     = 100 * generated / max(generated + rejected, 1)

print(f"\n{'='*60}")
print(f"CoT Generation Complete (STaR)")
print(f"{'='*60}")
print(f"  Total processed : {generated + rejected + errors}")
print(f"  Kept (correct)  : {generated}  ({keep_rate:.1f}%)")
print(f"  Rejected (wrong): {rejected}")
print(f"  Errors          : {errors}")
print(f"  Runtime         : {elapsed_total:.1f} min")
print(f"  Saved to        : {COT_CHECKPOINT}")
print()
print(f"  Per-category breakdown:")
print(f"  {'Category':<20} {'Kept':>6} {'Rejected':>9} {'Keep%':>7}")
print(f"  {'-'*46}")
for cat, s in sorted(cat_stats.items()):
    total_cat = s['kept'] + s['rejected']
    pct       = 100 * s['kept'] / max(total_cat, 1)
    print(f"  {cat:<20} {s['kept']:>6} {s['rejected']:>9} {pct:>6.0f}%")


No checkpoint found. Starting fresh.

Generation queue: 7529 remaining  (of 7529 total eligible)
Batch size: 2  |  Max tokens: 300



## Cell 9 — Format Training Dataset

CoT examples get `<think>...</think>` format. Non-CoT examples stay as `\\boxed{answer}` but now use the correct chat template.

In [ ]:
# ── Lookup for fast CoT retrieval ────────────────────────────────────────────
cot_lookup = {d["id"]: d["cot"] for d in cot_data}
print(f"CoT lookup: {len(cot_lookup)} verified reasoning chains")

# ── Format functions ──────────────────────────────────────────────────────────
def format_cot_example(prompt: str, cot: str, answer: str) -> str:
    """Full CoT format: <think>reasoning</think>\n\\boxed{answer}.
    Uses apply_chat_template matching exactly what the evaluator does.
    """
    assistant = f"<think>\n{cot}\n</think>\n\\boxed{{{answer}}}"
    return apply_template(
        [{"role": "user", "content": prompt},
         {"role": "assistant", "content": assistant}],
    )

def format_plain_example(prompt: str, answer: str) -> str:
    """Plain format without CoT (roman_numerals and any non-CoT examples).
    Still uses correct chat template — improvement over Phase 1 format.
    """
    return apply_template(
        [{"role": "user", "content": prompt},
         {"role": "assistant", "content": f"\\boxed{{{answer}}}"}],
    )

# ── Build ─────────────────────────────────────────────────────────────────────
texts     = []
cot_count = 0
plain_count = 0

for _, row in train_df.iterrows():
    rid = str(row.get("id", ""))
    if rid in cot_lookup:
        text = format_cot_example(row["prompt"], cot_lookup[rid], str(row["answer"]))
        cot_count += 1
    else:
        text = format_plain_example(row["prompt"], str(row["answer"]))
        plain_count += 1
    texts.append(text)

train_df = train_df.copy()
train_df["text"] = texts

print(f"\nTraining data composition:")
print(f"  CoT examples   : {cot_count:>5}  ({100*cot_count/len(train_df):.1f}%)")
print(f"  Plain examples : {plain_count:>5}  ({100*plain_count/len(train_df):.1f}%)")
print(f"  Total          : {len(train_df)}")

# ── Token length check ────────────────────────────────────────────────────────
sample_texts   = texts[:200]
sample_lengths = [len(tokenizer.encode(t)) for t in sample_texts]
over_limit     = sum(1 for l in sample_lengths if l > MAX_SEQ_LENGTH)
print(f"\nToken length check (first 200 samples):")
print(f"  Min  : {min(sample_lengths)}")
print(f"  Max  : {max(sample_lengths)}")
print(f"  Avg  : {sum(sample_lengths)/len(sample_lengths):.0f}")
print(f"  Over {MAX_SEQ_LENGTH} tokens: {over_limit} ({100*over_limit/len(sample_lengths):.1f}%) — truncated")


## Cell 10 — Tokenize

In [24]:
# Reset padding side to right for training (left-pad was only for batched generation)
tokenizer.padding_side = "right"

train_dataset = Dataset.from_dict({"text": train_df["text"].tolist()})

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

train_dataset  = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
data_collator  = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f"Train dataset : {len(train_dataset)} tokenized examples")
print(f"CoT mix       : {cot_count} CoT ({100*cot_count/len(train_dataset):.1f}%) + "
      f"{plain_count} plain ({100*plain_count/len(train_dataset):.1f}%)")


Map:   0%|          | 0/9025 [00:00<?, ? examples/s]

Train dataset : 9025 tokenized examples
CoT mix       : 0 CoT (0.0%) + 9025 plain (100.0%)


## Cell 11 — Train (STaR: Phase 1 Adapter → Phase 2a)

**Key differences from Phase 1:**
| | Phase 1 | Phase 2a |
|---|---|---|
| Training format | `\\boxed{answer}` | `<think>cot</think>\n\\boxed{answer}` |
| Starting point | Base model | Phase 1 adapter (warm) |
| LR | 5e-5 | **2e-5** (lower — fine-tuning) |
| Epochs | 3 | **2** (warm start converges faster) |
| Seq length | 512 | **1024** (CoT needs room) |


In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,
    dataloader_num_workers=0,
    logging_steps=10,
    report_to="none",
    save_strategy="no",
    seed=RANDOM_SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print("Starting Phase 2a training (STaR warm start)...")
print(f"  Steps/epoch   : {steps_per_epoch}")
print(f"  Total steps   : {steps_per_epoch * NUM_EPOCHS}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  CoT mix        : {cot_count} CoT + {plain_count} plain")
print()

train_result = trainer.train()

print("\n✅ Training complete!")
print(f"  Final loss  : {train_result.training_loss:.4f}")
print(f"  Runtime     : {train_result.metrics.get('train_runtime', 0)/60:.1f} min")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting Phase 2a training (STaR warm start)...
  Steps/epoch   : 564
  Total steps   : 1128
  Effective batch: 16
  CoT mix        : 0 CoT + 9025 plain



Step,Training Loss
10,7.020922
20,6.043047
30,4.967023
40,3.593159
50,3.467564
60,2.958792
70,2.967352
80,3.776874
90,3.059113
100,2.740143


## Cell 12 — Evaluation (Competition Metric)

Uses `temperature=1.0`, `max_new_tokens=1024` and the actual `verify()` / `extract_final_answer()` functions — matching the competition evaluator exactly.

In [ ]:
print(f"Evaluating on {len(eval_df)} examples (competition metric)...")
print(f"  max_new_tokens : {EVAL_MAX_NEW_TOKENS}")
print(f"  temperature    : {EVAL_TEMPERATURE}  top_p={EVAL_TOP_P}")
print()

model.eval()
# Switch back to left-padding for inference
tokenizer.padding_side = "left"

predictions  = []
categories   = []
correct_list = []
eval_start   = time.time()

for i, (_, row) in enumerate(eval_df.iterrows()):
    # Append the \boxed{} instruction (matches what competition evaluator does)
    user_content = row["prompt"] + "\nPlease put your final answer inside `\\boxed{}`."
    prompt_text  = apply_template([{"role": "user", "content": user_content}],
                                  add_generation_prompt=True)

    inputs = tokenizer(
        prompt_text, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=EVAL_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=EVAL_TEMPERATURE,
            top_p=EVAL_TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = out_ids[0][inputs["input_ids"].shape[1]:]
    generated  = tokenizer.decode(new_tokens, skip_special_tokens=True)

    predicted  = extract_final_answer(generated)
    category   = classify_puzzle(row["prompt"], str(row["answer"]))
    is_correct = verify(str(row["answer"]), predicted)

    predictions.append(predicted)
    categories.append(category)
    correct_list.append(is_correct)

    if (i + 1) % 50 == 0 or (i + 1) == len(eval_df):
        acc_now = sum(correct_list) / len(correct_list)
        elapsed = (time.time() - eval_start) / 60
        print(f"  [{i+1:4d}/{len(eval_df)}]  acc={acc_now:.3f}  elapsed={elapsed:.1f}m")

print("\n✅ Inference complete.")


In [ ]:
eval_df = eval_df.copy()
eval_df["category"]  = categories
eval_df["predicted"] = predictions
eval_df["correct"]   = correct_list

overall_acc = sum(correct_list) / len(correct_list)
delta_local = overall_acc - 0.5611

print("=" * 65)
print("PHASE 2a EVALUATION RESULTS  (competition metric)")
print("=" * 65)
print(f"  Phase 1 local  : 0.5611  |  Phase 1 LB : 0.63")
print(f"  Phase 2a local : {overall_acc:.4f}  |  Delta vs P1 local: {delta_local:+.4f}")
print()

phase1_local = {"binary":0.4857,"decimal_math":0.5217,"integer_math":0.4000,
                "other":0.0298,"roman_numerals":1.0000,"word_sequence":0.6466}

print(f"{'Category':<20} {'P1 local':>9} {'P2a local':>10} {'Δ':>8}  CoT?")
print("-" * 58)
for cat in sorted(eval_df["category"].unique()):
    cdf     = eval_df[eval_df["category"] == cat]
    acc     = cdf["correct"].mean()
    p1      = phase1_local.get(cat, 0)
    d       = acc - p1
    has_cot = "✓" if cat in COT_CATEGORIES else "✗"
    n_cot   = cot_lookup and sum(1 for x in cot_data if x["category"] == cat)
    print(f"  {cat:<18} {p1:>9.4f} {acc:>10.4f} {d:>+8.4f}  {has_cot}")
print("-" * 58)
d_tot = overall_acc - 0.5611
print(f"  {'OVERALL':<18} {'0.5611':>9} {overall_acc:>10.4f} {d_tot:>+8.4f}")

no_found = sum(1 for p in predictions if p == 'NOT_FOUND')
print(f"\nFormat: {no_found} predictions returned NOT_FOUND ({100*no_found/len(predictions):.1f}%)")

errors_sample = eval_df[~eval_df["correct"]].head(5)
print("\nSample errors:")
for _, row in errors_sample.iterrows():
    print(f"  [{row['category']}] expected={row['answer']!r}  got={row['predicted']!r}")

# Save
results = {
    "phase": "phase2a_star_self_cot",
    "cot_generated": generated if 'generated' in dir() else len(cot_data),
    "cot_kept_after_star_filter": len(cot_data),
    "overall_accuracy": float(overall_acc),
    "phase1_local": 0.5611, "phase1_lb": 0.63,
    "per_category": {
        cat: {"accuracy": float(eval_df[eval_df["category"]==cat]["correct"].mean()),
              "n": int((eval_df["category"]==cat).sum())}
        for cat in sorted(eval_df["category"].unique())
    }
}
with open(f"{OUTPUT_DIR}/phase2a_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved: {OUTPUT_DIR}/phase2a_results.json")


## Cell 13 — Save Adapter & Package Submission

In [ ]:
print(f"Saving Phase 2a adapter → {ADAPTER_DIR}")
Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)

# Restore right-padding before saving tokenizer
tokenizer.padding_side = "right"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

files = sorted(Path(ADAPTER_DIR).glob("*"))
print(f"Files ({len(files)}):")
for f in files:
    print(f"  {f.name:<45} {f.stat().st_size/1024:>10.1f} KB")

cfg_path = Path(ADAPTER_DIR) / "adapter_config.json"
if cfg_path.exists():
    with open(cfg_path) as f:
        cfg = json.load(f)
    print(f"\n✅ adapter_config.json: rank={cfg.get('r')}  base={cfg.get('base_model_name_or_path','?')[:50]}")
else:
    print("\n❌ adapter_config.json MISSING")

# ── Zip for submission ────────────────────────────────────────────────────────
SUBMISSION_ZIP = f"{OUTPUT_DIR}/phase2a_submission.zip"
res = subprocess.run(
    ["zip", "-j", SUBMISSION_ZIP] + [str(f) for f in files],
    capture_output=True, text=True,
)
if res.returncode == 0:
    mb = Path(SUBMISSION_ZIP).stat().st_size / 1e6
    verify = subprocess.run(["unzip", "-l", SUBMISSION_ZIP], capture_output=True, text=True)
    print(f"\n✅ phase2a_submission.zip ({mb:.1f} MB)")
    print(verify.stdout)
    ok = "adapter_config.json" in verify.stdout
    print("✅ Ready to submit!" if ok else "❌ adapter_config.json missing from zip!")
else:
    print(f"❌ zip failed: {res.stderr}")


## ✅ Phase 2a Complete!

### Submit
1. Download `phase2a_submission.zip` from the output panel
2. Go to competition → Submit Predictions → upload the zip
3. Record the leaderboard score in `NOTES.md`

### Interpreting results
| Scenario | LB score | Interpretation |
|---|---|---|
| LB > 0.70 | 🎉 | Self-CoT highly effective, Gemini CoT should reach 0.80+ |
| LB 0.65–0.70 | ✅ | Solid improvement, CoT works, Gemini will do better |
| LB 0.63–0.65 | ⚠️ | Marginal gain, check per-category — binary/other improvement? |
| LB < 0.63 | ❌ | Regression — check STaR filter threshold, training format |

### Already running in parallel
While this notebook ran, `phase2_cot_generation.ipynb` should be generating
higher-quality CoT via Gemini API on your local WSL machine.
Phase 2b will use those Gemini CoT chains for a bigger accuracy jump.
